# Lean-42 — La conjecture de Poincaré en Lean : ce que dit l'énoncé, ce que la preuve mobilise

> **Série** : Lean · **Niveau** : avancé · **Prérequis** : Lean 4 de base, savoir lire un `import`
>
> Ce carnet fait partie de l'EPIC *origami géométrie différentielle* et suit le **Pli 2**.
> Il ne rejoue pas la preuve de la conjecture de Poincaré : cette fermeture a coûté
> 22 909 jobs et plusieurs heures de compilation en local. Il fait autre chose, qui est
> souvent plus instructif : il lit **ce que l'énoncé dit**, puis il **calcule** ce que la
> preuve mobilise.

L'idée directrice tient en une question. La preuve de la conjecture de Poincaré, dans le
dépôt amont que nous visitons, tient en **cinq lignes de tactiques**. Cinq lignes pour un
théorème qui a résisté un siècle. Où est passé le reste ?

Réponse courte : dans les `import`. Ce carnet construit le graphe d'imports du paquet et
mesure, chiffres à l'appui, l'écart entre la **brièveté de l'énoncé** et la **taille de ce
qu'il faut pour l'énoncer**.

## Prérequis d'exécution

Le carnet lit les sources d'un dépôt tiers, `qinz1yang/differential-geometry`, épinglé au
tag `v0.1.3` (commit `7a48598d35109aa99d1cc678e2724c213cdf4ff3`, licence Apache-2.0).
Aucune source de l'amont n'est recopiée ici : le carnet ne fait que **lire** un clone ou un
paquet Lake déjà présent sur la machine.

La cellule suivante cherche ce paquet à plusieurs emplacements. Si aucun n'existe, elle
s'arrête avec un message explicite : il faut alors cloner le dépôt amont au tag `v0.1.3`,
ou renseigner `DIFFERENTIAL_GEOMETRY_SRC` avec le chemin de sa racine.


## 1. Trouver les sources amont

Trois emplacements sont essayés, dans l'ordre : la variable d'environnement
`DIFFERENTIAL_GEOMETRY_SRC`, puis un clone voisin, puis le paquet téléchargé par Lake.


In [1]:
import os
from pathlib import Path

TAG_ATTENDU = "v0.1.3"

# Emplacements candidats, du plus explicite au plus implicite.
candidats = []
if os.environ.get("DIFFERENTIAL_GEOMETRY_SRC"):
    candidats.append(Path(os.environ["DIFFERENTIAL_GEOMETRY_SRC"]))
candidats += [
    Path("DifferentialGeometry"),
    Path("differential_lean/.lake/packages/DifferentialGeometry"),
    Path(".lake/packages/DifferentialGeometry"),
    Path.home() / ".lake/packages/DifferentialGeometry",
]

RACINE = next((c for c in candidats if (c / "DifferentialGeometry").is_dir()), None)
if RACINE is None:
    raise FileNotFoundError(
        "Sources amont introuvables. Cloner qinz1yang/differential-geometry au tag "
        f"{TAG_ATTENDU}, ou renseigner DIFFERENTIAL_GEOMETRY_SRC.\n"
        "Emplacements essayes : " + ", ".join(str(c) for c in candidats)
    )

# On affiche le chemin tel qu'il a ete trouve, pas sa forme absolue : une sortie
# committee ne doit pas dependre de la machine qui l'a produite.
print("racine amont :", RACINE)
print("existe       :", (RACINE / "DifferentialGeometry").is_dir())

racine amont : differential_lean\.lake\packages\DifferentialGeometry
existe       : True


## 2. Ce que dit l'énoncé

On lit d'abord le module qui déclare la conjecture, **tel quel**, sans le paraphraser. C'est
lui qui fait autorité : tout le reste du carnet parle de ses `import`, pas de nos souvenirs.


In [2]:
MODULE_ENONCE = RACINE / "DifferentialGeometry/Topology/ThreeManifold/Poincare.lean"
source = MODULE_ENONCE.read_text(encoding="utf-8")
lignes = source.splitlines()

print(f"{MODULE_ENONCE.name} — {len(lignes)} lignes")
print("-" * 70)
for numero, ligne in enumerate(lignes, 1):
    print(f"{numero:3d} | {ligne}")

# Longueur reelle de la preuve : du `theorem` jusqu'a la ligne `end` du namespace.
debut = next(i for i, l in enumerate(lignes) if l.startswith("theorem poincare_conjecture"))
fin = next(i for i, l in enumerate(lignes[debut:], debut) if l.startswith("end"))
bloc = lignes[debut:fin]
i_by = next(i for i, l in enumerate(bloc) if l.rstrip().endswith(":= by"))
apres_by = [l for l in bloc[i_by + 1:] if l.strip()]

print()
print("bloc du theoreme, enonce compris :", len(bloc), "lignes")
print("apres `:= by`, hors lignes vides :", len(apres_by), "lignes")

Poincare.lean — 23 lignes
----------------------------------------------------------------------
  1 | import DifferentialGeometry.Topology.PiecewiseLinear.Moise352Producer
  2 | import DifferentialGeometry.Geometry.Flow.RicciFlow.Surgery.Skeleton.PoincareEndgame
  3 | 
  4 | set_option autoImplicit false
  5 | 
  6 | open scoped Manifold ContDiff
  7 | 
  8 | namespace DifferentialGeometry.Topology
  9 | 
 10 | universe u
 11 | 
 12 | theorem poincare_conjecture
 13 |     (M : Type u) [TopologicalSpace M]
 14 |     [ChartedSpace (EuclideanSpace ℝ (Fin 3)) M]
 15 |     [T2Space M] [CompactSpace M] [SimplyConnectedSpace M] :
 16 |     Nonempty (M ≃ₜ Metric.sphere (0 : EuclideanSpace ℝ (Fin 4)) 1) := by
 17 |   obtain ⟨C, hC⟩ := PiecewiseLinear.exists_isManifold_three (M := M)
 18 |   let := C
 19 |   let : IsManifold (𝓡 3) ∞ M := hC
 20 |   obtain ⟨f⟩ := PDE.RicciFlow.Surgery.Topology.smoothPoincareConjecture_holds M
 21 |   exact ⟨f.toHomeomorph⟩
 22 | 
 23 | end DifferentialGeometry.T

### Lecture du résultat

Le module fait deux choses, et il vaut la peine de les séparer.

Il **importe deux branches** qui n'ont rien à voir l'une avec l'autre : le découpage
PL de Moise d'un côté, la fin de partie du flot de Ricci de l'autre. Et il **prouve** :
la cellule précédente a compté les lignes de la preuve, et le compte est dans sa sortie.

Les temps de cette preuve ne sont pas symétriques :

1. **Moise est la porte d'entrée.** La conjecture topologique ne suppose qu'une structure
   de variété topologique ; le flot de Ricci, lui, exige une structure **lisse**. C'est
   `exists_isManifold_three` qui fabrique cette structure. Sans lui, la machine de Ricci
   n'a rien sur quoi mordre.
2. **Le moteur rend un difféomorphisme, pas un homéomorphisme.** `smoothPoincareConjecture_holds`
   conclut une équivalence *lisse* : c'est strictement plus fort que l'énoncé topologique.
3. **Le retour au topologique est une coercition.** `f.toHomeomorph` : tout l'écart entre
   les deux énoncés tient là.

Retenez cette asymétrie : elle explique pourquoi la preuve est courte **et** pourquoi elle
n'est pas triviale.


## 3. Le graphe d'imports comme donnée

On construit maintenant le graphe du paquet : chaque fichier `.lean` est un module, chaque
ligne `import` est une arête.

Le graphe est **calculé** ici, à partir des fichiers, et non recopié d'une figure de
référence. C'est le point de méthode du carnet : un chiffre qu'on ne sait pas recalculer
n'est pas une mesure, c'est une citation.


In [3]:
import re
from collections import Counter

PAQUET = RACINE / "DifferentialGeometry"

imports = {}      # module -> liste de modules importes
lignes_de = {}    # module -> nombre de lignes du fichier

for chemin in sorted(PAQUET.rglob("*.lean")):
    relatif = chemin.relative_to(RACINE).as_posix()
    module = relatif[:-len(".lean")].replace("/", ".")
    texte = chemin.read_text(encoding="utf-8", errors="ignore")

    deps = []
    for ligne in texte.splitlines():
        ligne = ligne.strip()
        if ligne.startswith("import "):
            deps.extend(ligne.split()[1:])
    imports[module] = deps
    lignes_de[module] = len(texte.splitlines())

print("modules du paquet :", f"{len(imports):,}")
print("lignes du paquet  :", f"{sum(lignes_de.values()):,}")

internes = sum(1 for m in imports for d in imports[m] if d in imports)
print("aretes internes   :", f"{internes:,}")
print("aretes vers Mathlib ou hors paquet :",
      f"{sum(len(imports[m]) for m in imports) - internes:,}")

modules du paquet : 17,555
lignes du paquet  : 4,704,458
aretes internes   : 52,908
aretes vers Mathlib ou hors paquet : 11,291


### Lecture du résultat

Deux comptes, et un troisième qui surprend souvent : les arêtes **internes** au paquet et
celles qui sortent vers Mathlib ne sont pas du même ordre.

Un module qui importe Mathlib importe un monde déjà compilé, partagé par tout Lean. Un
module qui importe un module du paquet s'ajoute au travail de celui qui voudra compiler
l'ensemble. C'est cette seconde catégorie qui rend la clôture coûteuse, et c'est elle que
la section suivante mesure.


## 4. La clôture de l'énoncé

La **clôture** d'un module est l'ensemble des modules qu'il faut compiler pour qu'il compile,
transitivement. C'est une fermeture transitive ordinaire : un parcours en largeur suffit.

Une convention, écrite plutôt que silencieuse : les imports **hors du paquet** (Mathlib) ne
sont pas comptés dans la clôture. On mesure le paquet visité, pas la totalité de
l'écosystème Lean.


In [4]:
from collections import deque

CIBLE = "DifferentialGeometry.Topology.ThreeManifold.Poincare"


def cloture(cible):
    """Fermeture transitive des imports, restreinte au paquet."""
    vus = {cible}
    file = deque([cible])
    while file:
        module = file.popleft()
        for dep in imports.get(module, []):
            if dep in imports and dep not in vus:
                vus.add(dep)
                file.append(dep)
    return vus


cloture_enonce = cloture(CIBLE)
lignes_cloture = sum(lignes_de[m] for m in cloture_enonce)

print("modules dans la clôture :", f"{len(cloture_enonce):,}", f"/ {len(imports):,}")
print("lignes dans la clôture  :", f"{lignes_cloture:,}", f"/ {sum(lignes_de.values()):,}")
print("part des modules        :", f"{100 * len(cloture_enonce) / len(imports):.1f} %")
print("part des lignes         :", f"{100 * lignes_cloture / sum(lignes_de.values()):.1f} %")

modules dans la clôture : 14,196 / 17,555
lignes dans la clôture  : 4,020,908 / 4,704,458
part des modules        : 80.9 %
part des lignes         : 85.5 %


### Lecture du résultat

L'énoncé mobilise la **très grande majorité** du paquet — les deux pourcentages affichés
ci-dessus le disent mieux qu'une phrase. C'est le cœur du carnet : treize lignes de preuve,
mais presque tout un dépôt de géométrie différentielle derrière elles.

Ce n'est pas une anomalie. C'est la signature d'une mathématique **cumulative** : la
conjecture ne se démontre pas avec ce qu'on écrit dans son fichier, elle se démontre avec
tout ce que la communauté a formalisé avant.


## 5. Composition de la clôture

Un total ne dit pas de quoi il est fait. On ventile la clôture par famille de modules.


In [5]:
familles = Counter()
familles_lignes = Counter()
for module in cloture_enonce:
    famille = ".".join(module.split(".")[:2])
    familles[famille] += 1
    familles_lignes[famille] += lignes_de.get(module, 0)

print(f"{'famille':<42} {'modules':>8} {'lignes':>12}")
print("-" * 66)
for famille, compte in familles.most_common():
    print(f"{famille:<42} {compte:>8,} {familles_lignes[famille]:>12,}")

famille                                     modules       lignes
------------------------------------------------------------------
DifferentialGeometry.Geometry                 6,050    1,847,820
DifferentialGeometry.Topology                 4,813      888,734
DifferentialGeometry.Analysis                 2,809    1,087,999
DifferentialGeometry.External                   360      156,329
DifferentialGeometry.Tensor                     107       29,072
DifferentialGeometry.Bundle                      57       10,954


### Lecture du résultat

La clôture n'est pas répartie au hasard. La géométrie domine en volume, mais la topologie
pèse lourd en nombre de modules — et l'analyse, qu'on n'attendrait pas dans un énoncé de
topologie, est présente parce que le flot de Ricci est une **équation aux dérivées
partielles** avant d'être un objet géométrique.

C'est un fait pédagogique utile : un énoncé de topologie se démontre avec de l'analyse.


## 6. Les maillons de la chaîne, mesurés séparément

La preuve traverse trois mondes : le découpage PL de Moise, le flot de Ricci avec chirurgie,
l'extinction en temps fini. On mesure la clôture de chacun, séparément, pour voir lequel
coûte le plus.


In [6]:
MAILLONS = [
    ("Découpage PL (Moise)",
     "DifferentialGeometry.Topology.PiecewiseLinear.Moise352Producer"),
    ("Flot de Ricci avec chirurgie",
     "DifferentialGeometry.Geometry.Flow.RicciFlow.Surgery.Skeleton.PoincareEndgame"),
    ("Extinction en temps fini",
     "DifferentialGeometry.Geometry.Flow.RicciFlow.DimensionThree.PositiveRicci.Extinction"),
]

print(f"{'maillon':<32} {'modules':>8} {'lignes':>12}")
print("-" * 56)
for libelle, module in MAILLONS:
    if module not in imports:
        print(f"{libelle:<32} {'ABSENT':>8}")
        continue
    c = cloture(module)
    print(f"{libelle:<32} {len(c):>8,} {sum(lignes_de[m] for m in c):>12,}")

maillon                           modules       lignes
--------------------------------------------------------
Découpage PL (Moise)                3,507      720,934
Flot de Ricci avec chirurgie       11,348    3,521,645
Extinction en temps fini            2,399    1,282,938


### Lecture du résultat

Les trois maillons ne pèsent pas le même poids, et l'ordre n'est pas celui du récit.

Le découpage PL est le plus modeste en modules : c'est de la topologie combinatoire, elle
s'appuie sur peu de choses extérieures. Le flot de Ricci avec chirurgie est de loin le plus
lourd : il traîne l'analyse, les EDP, la géométrie riemannienne. L'extinction est
intermédiaire, mais elle est déjà grande à elle seule.

Autrement dit, le coût de la preuve n'est pas réparti là où le récit la place. Le récit dit
« Moise, puis Ricci » ; la mesure dit que le second maillon porte presque tout.


## 7. Les pivots : ce qui est importé par tout le monde

Un module **pivot** est un module importé par beaucoup d'autres modules de la clôture. On
classe par *in-degree* — le nombre de modules de la clôture qui l'importent.

Ces pivots ne sont presque jamais des théorèmes célèbres. Ce sont des **outils**.


In [7]:
indegre = Counter()
for module in cloture_enonce:
    for dep in imports.get(module, []):
        if dep in imports:
            indegre[dep] += 1

print(f"{'module pivot':<62} {'importé par':>12}")
print("-" * 76)
for module, compte in indegre.most_common(10):
    print(f"{module:<62} {compte:>12,}")

module pivot                                                    importé par


----------------------------------------------------------------------------
DifferentialGeometry.Geometry.Metric.Coordinates.ChartGram               89
DifferentialGeometry.Geometry.Metric.TensorInner.Tensor0S.Coordinates.MetricComparison           86
DifferentialGeometry.Topology.PiecewiseLinear.GeneralPosition            69
DifferentialGeometry.Geometry.Operator.Gradient.Basic                    68
DifferentialGeometry.Analysis.Spectral.Intrinsic.DeTurck.Remainder.Defs           67
DifferentialGeometry.Geometry.Metric.PointwiseInner.Bounds               66
DifferentialGeometry.Analysis.Integration.Measure.Riemannian.Properties           65
DifferentialGeometry.Geometry.Metric.Comparison.DistanceScaling           64
DifferentialGeometry.Topology.Manifold.PartialDiffeomorph.Opens           64
DifferentialGeometry.Analysis.Integration.Measure.Chart.Density           61


### Lecture du résultat

Le sommet du classement est un module de coordonnées de carte et de comparaison de métriques :
un outil de calcul local, pas un énoncé profond. C'est la forme habituelle d'un socle
formel — les résultats célèbres sont au sommet du graphe, les outils qui les rendent
possibles sont au **centre**, et ce sont eux qui sont importés par tout le monde.

Question à garder en tête pour l'exercice 2 : ce classement dépend-il de la cible ?


## 8. Exercices

Trois marches à refaire soi-même. Chacune mesure quelque chose que les sections précédentes
affirment, et le but est de le vérifier plutôt que de nous croire.

### Exercice 1 — la clôture d'un autre énoncé

Recalculer la clôture de Bonnet–Myers
(`DifferentialGeometry.Geometry.Comparison.BonnetMyers.Diameter`) et la comparer à celle de
Poincaré, en modules et en lignes.

La compétence visée : la clôture est une propriété de **l'énoncé**, pas du fichier qui le
déclare. Deux énoncés peuvent habiter le même dépôt et ne pas traîner la même chose.


In [8]:
# Exercice 1 : clôture de Bonnet-Myers, comparée à celle de Poincaré.
BONNET_MYERS = "DifferentialGeometry.Geometry.Comparison.BonnetMyers.Diameter"


def cloture_bonnet_myers():
    # TODO etudiant : calculer la clôture de BONNET_MYERS avec la fonction `cloture`
    # définie en section 4, puis renvoyer un dict avec au moins :
    #   {"modules": ..., "lignes": ..., "part_modules": ..., "part_lignes": ...}
    # Comparer enfin à `cloture_enonce` et `lignes_cloture` (section 4).
    return None


resultat = cloture_bonnet_myers()
print("Exercice 1 a completer — clôture de Bonnet-Myers :", resultat)

Exercice 1 a completer — clôture de Bonnet-Myers : None


### Exercice 2 — le pivot dépend de la cible

Trouver le module de plus fort in-degree de la clôture **de Bonnet–Myers**, et le comparer
au premier du classement de la section 7 (qui était calculé pour Poincaré).

La compétence visée : un « module central » n'existe pas dans l'absolu. Ce qui est central
est central *pour une cible donnée*.


In [9]:
# Exercice 2 : pivot de plus fort in-degree, pour deux cibles différentes.
def pivot(cible):
    # TODO etudiant : calculer l'in-degree restreint à la clôture de `cible`,
    # puis renvoyer le couple (module, nombre d'imports) de tête.
    return None


print("Exercice 2 a completer — pivot pour Poincaré    :", pivot(CIBLE))
print("Exercice 2 a completer — pivot pour Bonnet-Myers :", pivot(BONNET_MYERS))

Exercice 2 a completer — pivot pour Poincaré    : None
Exercice 2 a completer — pivot pour Bonnet-Myers : None


### Exercice 3 — le chemin le plus court n'est pas la dépendance

Depuis l'énoncé `poincare_conjecture`, trouver le **plus court chemin d'imports** vers le
module d'extinction
(`DifferentialGeometry.Geometry.Flow.RicciFlow.DimensionThree.PositiveRicci.Extinction`),
et l'afficher comme une suite de modules.

Puis confronter ce chemin à la phrase mathématique : Moise sert à *entrer* dans le monde
lisse, l'extinction est une *étape du moteur*. Le chemin d'imports dit-il cet ordre ?

La compétence visée — la plus importante du carnet : un graphe d'imports porte la
**compilation**, pas l'**argument**. Les deux se ressemblent souvent, et diverger est
instructif.


In [10]:
from collections import deque

ARRIVEE = ("DifferentialGeometry.Geometry.Flow.RicciFlow"
           ".DimensionThree.PositiveRicci.Extinction")


def plus_court_chemin(depart, arrivee):
    # TODO etudiant : parcours en largeur sur `imports` (arêtes = imports internes au paquet),
    # renvoyer la liste des modules du plus court chemin, de `depart` à `arrivee`.
    # Indice : conserver un dictionnaire `parent` comme en section 4.
    return None


chemin = plus_court_chemin(CIBLE, ARRIVEE)
print("Exercice 3 a completer — plus court chemin :", chemin)

Exercice 3 a completer — plus court chemin : None


## 9. Conclusion — ce que ce carnet laisse au lecteur

Trois choses, dans l'ordre d'importance.

**Un énoncé court n'est pas une preuve courte.** Cinq lignes de tactiques — le compte est
dans la sortie de la section 2 — et presque tout un dépôt de géométrie différentielle
derrière elles. La brièveté d'un énoncé formel mesure la qualité de l'interface, pas le
travail.

**Un graphe d'imports est une donnée, pas une illustration.** On l'a construit, on l'a
interrogé, on en a tiré des chiffres. C'est reproductible, et c'est réfutable : toute personne
qui refait les cellules doit retrouver les mêmes comptes au même commit.

**Les outils sont au centre, les théorèmes au sommet.** Les modules les plus importés ne sont
pas les plus célèbres. C'est vrai dans ce dépôt comme dans la plupart des bibliothèques
formelles — et c'est une bonne nouvelle pédagogique : ce qui se réutilise n'est pas ce qui
se raconte.

### Ce que le carnet ne fait pas

Il ne vérifie pas la conjecture, et il ne prétend pas le faire. Il lit la **dépendance
axiomatique** de l'énoncé telle que l'amont la publie, via le témoin `#print axioms` du
fichier `PoincareChain.lean` du lake compagnon. Ce témoin dit qu'aucun axiome prohibé
(`sorryAx`, `native_decide.*`) n'apparaît dans la chaîne. Il ne dit rien de la vérité
mathématique de l'énoncé, ni de la qualité de la formalisation.

### Pour aller plus loin

Le **Pli 3** de l'EPIC prendra les carnets que le Pli 2 laisse de côté : le lemme de Morse,
la cohomologie de de Rham, Bonnet–Myers comme objet d'étude et non comme cible d'exercice, le
principe du maximum, et la monotonie de Perelman — l'outil de non-extinction, qui mérite son
carnet à lui seul.

Le **Pli 4** confrontera ce dépôt à nos propres lakes : `knot_lean`, `grothendieck_lean`,
`conway_lean`.
